# RikiGaku-AHIRU（AHIRU-μTJ42V）最先端航空宇宙工学グレード・マルチフィジックス検証環境（MDAO / V&V）

本ノートブックは、**簡易計算・モック・ダミー値を完全禁止**し、実ソルバー群（`python-fcl`, `Cantera`, `OpenMDAO`, `Gmsh`, `SU2_CFD`, `CalculiX (ccx)`, `ROSS 3.0`, `PySpice/Ngspice`, `CasADi`, `MuJoCo`, `JAX GPU`, `OpenTURNS`, `SALib`, `Plotly`）を **JSON + HDF5 チェックポイント経由で完全連成（Coupled MDAO）** させた航空宇宙グレード V&V ＆ 自動設計最適化システムです。

### 連成パイプライン（Phase 0 → Phase 9）
```
[Phase 0] 実ソルバー一括導入・Google Driveチェックポイント・GPU/CPU診断
   ↓
[Phase 1] python-fcl 静的干渉体積・±14°スウェプト動的衝突・CTE熱膨張チップクリアランス
   ↓ (exact CAD areas, CoG, inertia, hot tip clearances)
[Phase 2] Cantera 詳細燃焼動力学(n-Dodecane: cp(T), T_ad, S_L, τ_ign, LBO) + OpenMDAO 非設計点サイクル
   ↓ (real-gas gamma/cp/mu, Pt/Tt, thrust, TSFC)
[Phase 3] Gmsh 境界層メッシュ + mpirun SU2_CFD (RANS k-ω SST): 吸気DC60 / IGE・HGI地面効果 / 外形空力 + GCI
   ↓ (ΔT_HGI, CD/CL/Cm/CoP)
[Phase 4] CalculiX (ccx) 3D過渡熱解析 (伝導+対流+Stefan-Boltzmann輻射+接触熱抵抗) + 停止後300秒ヒートソークバック
   ↓ (nodal temperatures T_liner, T_battery, T_frame)
[Phase 5] CalculiX (ccx) 100%/125%バースト熱応力連成3D FEA + ROSS 3.0 キャンベル/危険速度/ISO 1940/ジャイロ軸受荷重
   ↓ (H_rotor, bearing loads, structural SF)
[Phase 6] PySpice / Ngspice 温度連成ESR・始動突入ブラウンアウト・サージ・10ms以内フェイルセーフ遮断
   ↓ (bus/servo power margins)
[Phase 7] CasADi NMPC + LQR + MuJoCo 接触着地 + JAX GPU 並列 6-DOF 非線形モンテカルロ (燃料移動・スロッシング・ジャイロ・片側RCS異常)
   ↓ (flight crash rate, 3σ attitude error)
[Phase 8] OpenTURNS 多項式カオス展開(PCE) 破壊確率 Pf + SALib Sobol グローバル感度解析 (S1 / ST)
   ↓ (Pf, reliability %, top critical tolerances)
[Phase 9] Plotly インタラクティブV&Vダッシュボード + OpenMDAO SLSQP 自動最適設計 & 実ソルバー再検証
```

**ワンクリック実行**: ランタイムを **A100 GPU（または L4 GPU）** に設定し、**`ランタイム` → `すべてのセルを実行`（`Ctrl+F9`）** をクリックしてください。


In [ ]:
#@title 1. ユーザー設計仕様（YAML編集可能 — デフォルトは RikiGaku-AHIRU 実測諸元）
REPO_URL = "https://github.com/sc8z35a-collab/RikiGaku-AHIRU.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
RESUME_FROM_CHECKPOINT = False #@param {type:"boolean"}

USER_SPEC_YAML = """
vehicle_spec:
  name: "RikiGaku-AHIRU Rev.2 (AHIRU-uTJ42V Flight-Ready)"
  dimensions_mm: { length: 180.0, width: 100.0, height: 200.0 }
  mass_g: { dry_mass: 460.6, fuel_mass: 111.6, payload_mass: 0.0 }
  cad_step_or_stl_path: ""   # 空欄時はリポジトリ index.html から58パーツ3D CADをビット完全自動抽出
propulsion_spec:
  fuel_type: "Jet-A1 / Kerosene + 5% Oil (n-Dodecane C12H26 surrogate)"
  max_thrust_N: 11.43
  max_rpm: 201000
  idle_rpm: 68000
  compressor_diameter_mm: 38.0
  turbine_diameter_mm: 36.4
  pressure_ratio: 2.82
  thrust_vectoring_deg: 14.0
  rcs_type: "ブリードエア"
avionics_and_power:
  battery: "3S 705mAh 60C LiHV"
  battery_cells_s: 3
  battery_mah: 705.0
  imu_rate_hz: 1000
  servo_bandwidth_hz: 15.0
"""
print("[SPEC] User YAML configured successfully.")


In [ ]:
#@title 2. Phase 0：Google Colab 全検証ソルバー一括セットアップ＆動作テスト
import os, sys, shutil, subprocess, multiprocessing, json
from pathlib import Path

# 1. Google Driveマウント & チェックポイント保存先作成
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    WORK_DIR = Path('/content/drive/MyDrive/Aero_MDAO_Verification')
except Exception:
    WORK_DIR = Path('/content/Aero_MDAO_Verification')

WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / 'checkpoints').mkdir(exist_ok=True)
(WORK_DIR / 'meshes').mkdir(exist_ok=True)
(WORK_DIR / 'reports').mkdir(exist_ok=True)
print(f"[INFO] 作業ディレクトリ: {WORK_DIR}")

# 2. リポジトリ同期 (index.html + mdao_vv モジュール群)
REPO_DIR = Path("/content/RikiGaku-AHIRU")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
for _m in [k for k in sys.modules if k == 'mdao_vv' or k.startswith('mdao_vv.')]:
    del sys.modules[_m]

# 3. ハードウェア資源の自動検出
CPU_CORES = multiprocessing.cpu_count()
gpu_info = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                          shell=True, capture_output=True, text=True)
GPU_AVAILABLE = (gpu_info.returncode == 0 and len(gpu_info.stdout.strip()) > 0)
print(f"[HW] CPU Cores: {CPU_CORES} | GPU: {gpu_info.stdout.strip() if GPU_AVAILABLE else 'CPU Only'}")

# 4. システム系ソルバー導入 (CalculiX FEA, Ngspice回路, Gmsh, OpenMPI, FCL, Node.js)
print("[INSTALL] システムソルバー群をインストール中...")
apt_cmd = """
apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
    calculix-ccx ngspice libngspice0 libngspice0-dev \
    openmpi-bin libopenmpi-dev gmsh libglu1-mesa \
    libfcl-dev liboctomap-dev libeigen3-dev libboost-all-dev \
    nodejs curl bzip2 unzip > /dev/null 2>&1
"""
subprocess.run(apt_cmd, shell=True, check=True)

# 5. Micromamba経由でCLI航空宇宙CFDソルバー (SU2) を独立バイナリ導入 (Colab Pythonと非衝突)
MAMBA_BIN = "/usr/local/bin/micromamba"
SU2_ENV = "/opt/su2_env"
if not Path(f"{SU2_ENV}/bin/SU2_CFD").exists():
    print("[INSTALL] SU2 (圧縮性RANS CFDソルバー) を高速バイナリ導入中...")
    mamba_setup = f"""
    curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /usr/local/bin/ --strip-components=1 bin/micromamba > /dev/null 2>&1
    {MAMBA_BIN} create -y -q -p {SU2_ENV} -c conda-forge su2 > /dev/null 2>&1
    """
    subprocess.run(mamba_setup, shell=True, check=True)
# IMPORTANT: Append SU2 bin so Colab python3 is NOT shadowed; mdao_vv automatically pairs SU2_CFD with its sibling mpirun!
if f"{SU2_ENV}/bin" not in os.environ["PATH"]:
    os.environ["PATH"] = os.environ["PATH"] + f":{SU2_ENV}/bin"

# 6. Python航空宇宙・力学・制御・回路・UQライブラリ一括導入 (ColabのCUDA対応JAXは温存!)
print("[INSTALL] Pythonマルチフィジックスライブラリ群をインストール中...")
pip_packages = [
    "cantera",
    "openmdao",
    "ross-rotordynamics",
    "casadi",
    "control",
    "mujoco",
    "PySpice",
    "openturns",
    "SALib",
    "gmsh",
    "trimesh",
    "python-fcl",
    "rtree",
    "meshio",
    "plotly",
    "pyyaml", "h5py", "scipy"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pip_packages, check=True)
# Only install jax if not already present (preserves Colab Pro A100 jax[cuda12]!)
try:
    import jax
    print(f"[HW] Preinstalled JAX preserved: {jax.__version__} ({jax.default_backend()})")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax[cuda12]" if GPU_AVAILABLE else "jax"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cadquery"], check=False)

# 7. 全ソルバー導入・起動スモークテスト
print("\n=== 全検証ソルバー導入チェック ===")
cli_tools = {"SU2_CFD": "SU2_CFD", "CalculiX (ccx)": "ccx", "Ngspice": "ngspice", "OpenMPI": "mpirun", "Node.js": "node"}
for label, cmd in cli_tools.items():
    found = shutil.which(cmd)
    print(f"  [{'OK' if found else 'NG'}] {label:18s}: {found}")

py_modules = ["cantera", "openmdao", "ross", "casadi", "control", "mujoco", "jax",
              "PySpice", "openturns", "SALib", "gmsh", "trimesh", "fcl", "plotly"]
for mod in py_modules:
    try:
        __import__(mod)
        print(f"  [OK] Python:{mod}")
    except Exception as e:
        print(f"  [NG] Python:{mod} -> {e}")

from mdao_vv.state_io import MDAOStateStore, detect_hpc_env, load_user_spec
STORE = MDAOStateStore(WORK_DIR)
ENV_INFO = detect_hpc_env(WORK_DIR)
SPEC = load_user_spec(USER_SPEC_YAML)
STORE.save_phase("env_config", ENV_INFO)
STORE.save_phase("user_spec", SPEC)
print("\n[READY] Phase 0 完了。Phase 1以降へ進行可能です。")


In [ ]:
#@title 3. Phase 1：3D幾何整合・python-fcl 静的/スウェプト動的衝突・CTE熱膨張クリアランス
from mdao_vv.phase01_geometry_fcl import run_phase01

if RESUME_FROM_CHECKPOINT and STORE.has_phase("phase01_geometry"):
    res_p1 = STORE.load_phase("phase01_geometry")
    print("[RESUME] Loaded phase01_geometry from checkpoint.")
else:
    res_p1 = run_phase01(SPEC, STORE, REPO_DIR)

for k, c in res_p1["checks"].items():
    print(f"  [{'PASS' if c['pass'] else 'FAIL'}] {k:32s}: {c['measured']}")


In [ ]:
#@title 4. Phase 2：Cantera 詳細燃焼化学動力学（Jet-A1サロゲート）& OpenMDAO 非設計点サイクル
from mdao_vv.phase02_cantera_openmdao import run_phase02

if RESUME_FROM_CHECKPOINT and STORE.has_phase("phase02_propulsion"):
    res_p2 = STORE.load_phase("phase02_propulsion")
    print("[RESUME] Loaded phase02_propulsion from checkpoint.")
else:
    res_p2 = run_phase02(SPEC, STORE)

for k, c in res_p2["checks"].items():
    print(f"  [{'PASS' if c['pass'] else 'FAIL'}] {k:32s}: {c['measured']}")


In [ ]:
#@title 5. Phase 3：Gmsh 境界層メッシュ + mpirun SU2_CFD (RANS k-ω SST) 空力・地面効果IGE/HGI・GCI
from mdao_vv.phase03_gmsh_su2_cfd import run_phase03

if RESUME_FROM_CHECKPOINT and STORE.has_phase("phase03_cfd_su2"):
    res_p3 = STORE.load_phase("phase03_cfd_su2")
    print("[RESUME] Loaded phase03_cfd_su2 from checkpoint.")
else:
    res_p3 = run_phase03(SPEC, STORE, ENV_INFO)

for k, c in res_p3["checks"].items():
    print(f"  [{'PASS' if c['pass'] else 'FAIL'}] {k:32s}: {c['measured']}")


In [ ]:
#@title 6. Phase 4：CalculiX (ccx) 共役熱伝達・Stefan-Boltzmann輻射・停止後300秒ヒートソークバック過渡解析
from mdao_vv.phase04_calculix_thermal import run_phase04

if RESUME_FROM_CHECKPOINT and STORE.has_phase("phase04_thermal"):
    res_p4 = STORE.load_phase("phase04_thermal")
    print("[RESUME] Loaded phase04_thermal from checkpoint.")
else:
    res_p4 = run_phase04(SPEC, STORE)

for k, c in res_p4["checks"].items():
    print(f"  [{'PASS' if c['pass'] else 'FAIL'}] {k:32s}: {c['measured']}")


In [ ]:
#@title 7. Phase 5：CalculiX 100%/125%バースト熱応力連成FEA + ROSS 3.0 キャンベル・不釣り合い・ジャイロ軸受
from mdao_vv.phase05_calculix_ross import run_phase05

if RESUME_FROM_CHECKPOINT and STORE.has_phase("phase05_struct_rotor"):
    res_p5 = STORE.load_phase("phase05_struct_rotor")
    print("[RESUME] Loaded phase05_struct_rotor from checkpoint.")
else:
    res_p5 = run_phase05(SPEC, STORE)

for k, c in res_p5["checks"].items():
    print(f"  [{'PASS' if c['pass'] else 'FAIL'}] {k:32s}: {c['measured']}")


In [ ]:
#@title 8. Phase 6：PySpice / Ngspice 温度連成ESR・始動ブラウンアウト・サージ・10ms以内フェイルセーフ遮断
from mdao_vv.phase06_pyspice_ngspice import run_phase06

if RESUME_FROM_CHECKPOINT and STORE.has_phase("phase06_power_spice"):
    res_p6 = STORE.load_phase("phase06_power_spice")
    print("[RESUME] Loaded phase06_power_spice from checkpoint.")
else:
    res_p6 = run_phase06(SPEC, STORE)

for k, c in res_p6["checks"].items():
    print(f"  [{'PASS' if c['pass'] else 'FAIL'}] {k:32s}: {c['measured']}")


In [ ]:
#@title 9. Phase 7：CasADi NMPC/LQR + MuJoCo 接触着地 + JAX GPU 6-DOF 非線形モンテカルロ検証
from mdao_vv.phase07_casadi_mujoco_jax import run_phase07

if RESUME_FROM_CHECKPOINT and STORE.has_phase("phase07_gnc_mc"):
    res_p7 = STORE.load_phase("phase07_gnc_mc")
    print("[RESUME] Loaded phase07_gnc_mc from checkpoint.")
else:
    res_p7 = run_phase07(SPEC, STORE, ENV_INFO)

for k, c in res_p7["checks"].items():
    print(f"  [{'PASS' if c['pass'] else 'FAIL'}] {k:32s}: {c['measured']}")


In [ ]:
#@title 10. Phase 8：OpenTURNS 多項式カオス展開(PCE) 破壊確率 Pf & SALib Sobol グローバル感度解析
from mdao_vv.phase08_openturns_salib_uq import run_phase08

if RESUME_FROM_CHECKPOINT and STORE.has_phase("phase08_uq_sobol"):
    res_p8 = STORE.load_phase("phase08_uq_sobol")
    print("[RESUME] Loaded phase08_uq_sobol from checkpoint.")
else:
    res_p8 = run_phase08(SPEC, STORE)

for k, c in res_p8["checks"].items():
    print(f"  [{'PASS' if c['pass'] else 'FAIL'}] {k:32s}: {c['measured']}")
print("\nTop Critical Manufacturing Parameters (Sobol ST):")
for i, r in enumerate(res_p8["critical_parameter_ranking"], 1):
    print(f"  #{i} {r['parameter']:36s} | SALib ST={r['SALib_ST']:.4f} | OpenTURNS PCE ST={r['OpenTURNS_PCE_ST']:.4f}")


In [ ]:
#@title 11. Phase 9：統合V&V Plotlyダッシュボード & OpenMDAO SLSQP 自動最適設計＋実ソルバー再検証
from pathlib import Path
import zipfile
from IPython.display import HTML, display
from mdao_vv.phase09_dashboard_optimizer import run_phase09

res_p9 = run_phase09(SPEC, STORE)
print("\n=== FINAL MDAO / V&V VERDICT ===")
print(f"  Baseline CAD Verdict  : {res_p9['baseline_verdict']}")
print(f"  Optimized Redesign    : {res_p9['optimized_verdict']}")
if res_p9["openmdao_optimization"]:
    print("  OpenMDAO Optimal Vars :", res_p9["openmdao_optimization"]["design_vars"])
    print("  Live Re-Verification  :", res_p9["live_reverification"])

# Render interactive Plotly V&V HTML Dashboard inline in Colab!
dash_path = Path(res_p9["dashboard_html"])
display(HTML(dash_path.read_text(encoding="utf-8")))

# Package all checkpoints, HDF5, JSON, HTML dashboard & solver decks into ZIP
zip_path = WORK_DIR / "reports" / "ahiru_mdao_vv_bundle.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for sub in ("checkpoints", "reports"):
        for fp in (WORK_DIR / sub).rglob("*"):
            if fp.is_file() and fp.name != zip_path.name:
                zf.write(fp, arcname=str(fp.relative_to(WORK_DIR)))
print(f"\n[SAVED] Complete MDAO/V&V archive saved to: {zip_path} ({zip_path.stat().st_size/1024**2:.2f} MB)")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass
